# HSL City Bike Project: Station-Month Analysis Demo

This notebook provides a student-friendly exploratory demonstration of the **HSL City Bike Station-Month aggregated dataset** (`data/processed/station_month.csv`).

### Objectives
1. Load and inspect the cleaned `station_month.csv` dataset.
2. Examine seasonal ridership trends across the 2025 cycling season (April–October).
3. Identify top stations by ridership demand.
4. Analyze net flow (`returns - departures`) to discover bike rebalancing bottlenecks.
5. Visualize spatial distribution of bike stations across Helsinki and Espoo.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set plot aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

data_path = os.path.join('..', 'data', 'processed', 'station_month.csv')
df = pd.read_csv(data_path)
print(f"Loaded {len(df):,} station-month observations across {df['year_month'].nunique()} months.")
df.head()

## 1. Monthly Ridership Trend

In [ ]:
monthly_summary = df.groupby('year_month').agg(
    total_departures=('departures', 'sum'),
    active_stations=('station_id', 'nunique'),
    avg_trip_dist_km=('avg_distance_m', lambda x: (x.mean() / 1000).round(2)),
    avg_trip_dur_min=('avg_duration_sec', lambda x: (x.mean() / 60).round(1))
).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(monthly_summary['year_month'], monthly_summary['total_departures'] / 1000, color='#1f77b4', edgecolor='black')
ax.set_title('Monthly Total Trips (Thousands) - 2025 Cycling Season', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Departures (in thousands)')
plt.show()

display(monthly_summary)

## 2. Top 10 Busiest Stations

In [ ]:
top_stations = df.groupby(['station_id', 'station_name', 'city', 'capacity']).agg(
    total_departures=('departures', 'sum'),
    total_returns=('returns', 'sum'),
    net_flow=('net_flow', 'sum')
).reset_index().sort_values('total_departures', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_stations['station_name'], top_stations['total_departures'], color='#2ca02c', edgecolor='black')
ax.invert_yaxis()
ax.set_title('Top 10 Busiest Bike Stations by Total Departures (2025)', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Departures')
plt.show()

display(top_stations)

## 3. Station Flow Imbalances & Rebalancing Priorities

- **Surplus (`net_flow > 0`)**: More bikes arrive than depart. Docks risk becoming full, preventing rider drop-offs.
- **Deficit (`net_flow < 0`)**: More bikes depart than arrive. Docks risk becoming empty, leaving riders without bikes.

In [ ]:
all_stations = df.groupby(['station_id', 'station_name', 'city']).agg(
    total_departures=('departures', 'sum'),
    total_returns=('returns', 'sum'),
    net_flow=('net_flow', 'sum')
).reset_index()

top_surplus = all_stations.sort_values('net_flow', ascending=False).head(5)
top_deficit = all_stations.sort_values('net_flow', ascending=True).head(5)

imbalance = pd.concat([top_deficit, top_surplus]).sort_values('net_flow')
colors = ['#d62728' if v < 0 else '#1f77b4' for v in imbalance['net_flow']]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(imbalance['station_name'], imbalance['net_flow'], color=colors, edgecolor='black')
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.set_title('Key Rebalancing Hotspots: Extreme Surplus vs Deficit (2025)', fontsize=14, fontweight='bold')
ax.set_xlabel('Net Flow (Returns - Departures)')
plt.show()

## 4. Geographic Map of Bike Demand

In [ ]:
geo_df = df.dropna(subset=['latitude', 'longitude']).groupby(['station_id', 'station_name', 'latitude', 'longitude'])['departures'].sum().reset_index()

fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(
    geo_df['longitude'], 
    geo_df['latitude'], 
    c=geo_df['departures'], 
    cmap='viridis', 
    s=geo_df['departures'] / 150 + 10, 
    alpha=0.75, 
    edgecolor='none'
)
ax.set_title('Geographic Distribution of City Bike Stations (Helsinki & Espoo)', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Total Completed Departures (2025)')
plt.show()